# 🐼 Pandas Foundations for Data Analytics — E-commerce Edition
**Notebook 02: Pandas Basics for Beginners**

This notebook teaches you how to load CSVs, explore, clean, join, and aggregate data using Pandas — mirroring what you did in SQL.
Each section includes concepts, business context, runnable examples, and short exercises.
---

## 1️⃣ Importing Pandas & Loading Data
**Concepts:** `import pandas as pd`, `pd.read_csv()`, `.head()`, `.info()`, `.describe()`

**Business Context:** Analysts load CSV exports from databases/BI tools to begin analysis.

In [ ]:
import pandas as pd
pd.__version__  # show version

In [ ]:
# Adjust paths if running elsewhere. Recommended repo layout:
# ecom-python-portfolio/
# └─ data/
#    ├─ raw/
#    │   customers.csv, orders.csv, order_items.csv, products.csv
#    └─ processed/

ORDERS_PATH = '../data/raw/orders.csv'
CUSTOMERS_PATH = '../data/raw/customers.csv'
ORDER_ITEMS_PATH = '../data/raw/order_items.csv'
PRODUCTS_PATH = '../data/raw/products.csv'

try:
    orders = pd.read_csv(ORDERS_PATH)
    customers = pd.read_csv(CUSTOMERS_PATH)
    order_items = pd.read_csv(ORDER_ITEMS_PATH)
    products = pd.read_csv(PRODUCTS_PATH)
    print('Loaded CSVs successfully.')
    display(orders.head())
except FileNotFoundError as e:
    print('⚠️ Could not find one or more CSV files. Update the paths above to your environment.')
    print(e)

In [ ]:
orders.info() if 'orders' in globals() else None

**Exercise:** Load `products.csv` and display its first 5 rows.
*Hint:* `pd.read_csv(PRODUCTS_PATH)` and `.head()`

## 2️⃣ Exploring DataFrames
**Concepts:** `.shape`, `.columns`, `.dtypes`, `.value_counts()`

**Business Context:** Quick scans reveal missing data, category distribution, and schema.

In [ ]:
if 'orders' in globals():
    print('Orders shape:', orders.shape)
    print('Columns:', list(orders.columns))
    print('Dtypes:\n', orders.dtypes)
    if 'order_status' in orders.columns:
        print('\nOrder Status value counts:')
        print(orders['order_status'].value_counts())

**Exercise:** In `customers`, print the number of unique states (if you have a `state` column).
*Hint:* `customers['state'].nunique()`

## 3️⃣ Selecting Columns & Rows
**Concepts:** `df['col']`, `df[['col1','col2']]`, `.loc[]`, `.iloc[]`

**Business Context:** Extract relevant fields before aggregating or joining.

In [ ]:
if 'orders' in globals():
    orders_subset = orders[['order_id','customer_id','order_date','order_status']].head(10)
    display(orders_subset)

**Exercise:** Select only `customer_id`, `order_id`, and `order_date` from `orders` and show the first 10 rows.

## 4️⃣ Filtering Data
**Concepts:** Conditional filters with booleans; combine with `&` and `|`

**Business Context:** Focus on delivered orders, recent dates, or a specific region.

In [ ]:
if 'orders' in globals():
    delivered_orders = orders[orders['order_status'] == 'delivered']
    print('Delivered orders:', delivered_orders.shape[0])

**Exercise:** Filter orders placed after `'2024-01-01'`.
*Hint:* Convert `order_date` to datetime with `pd.to_datetime` first.

## 5️⃣ Data Cleaning & Preparation
**Concepts:** `.drop_duplicates()`, `.dropna()`, `pd.to_datetime()`, `.fillna()`

**Business Context:** Real data is messy — parse dates, remove dupes, standardize missing values.

In [ ]:
if 'orders' in globals():
    orders_clean = orders.copy()
    for c in ['order_date','shipped_date','delivered_date']:
        if c in orders_clean.columns:
            orders_clean[c] = pd.to_datetime(orders_clean[c], errors='coerce')
    before = orders_clean.shape[0]
    orders_clean = orders_clean.drop_duplicates()
    after = orders_clean.shape[0]
    print(f'Removed {before - after} duplicate rows.')
    if 'order_status' in orders_clean.columns:
        orders_clean['order_status'] = orders_clean['order_status'].fillna('unknown')
    orders_clean.head()

**Exercise:** In `customers`, show a null count per column using `.isnull().sum()`.

## 6️⃣ Joining Multiple Tables (SQL-style Joins)
**Concepts:** `pd.merge(left, right, on=, how=)`

**Business Context:** Combine orders with items and customers to compute KPIs like revenue and AOV.

In [ ]:
if 'orders' in globals() and 'order_items' in globals():
    orders_items = pd.merge(orders_clean, order_items, on='order_id', how='inner')
    display(orders_items.head())

**Exercise:** Left-join `order_items` with `products` on `product_id` to add product names.

## 7️⃣ Calculating KPIs
**Concepts:** Derived columns, arithmetic, `groupby()`, `.agg()`

**Business Context:** Generate total revenue, items per order, AOV — core business metrics.

In [ ]:
if 'orders' in globals() and 'order_items' in globals():
    orders_items = pd.merge(orders_clean, order_items, on='order_id', how='inner')
    orders_items['revenue'] = orders_items['quantity'] * orders_items['item_price']
    kpis = (
        orders_items.groupby('order_id')
        .agg(total_revenue=('revenue','sum'), total_items=('quantity','sum'))
        .reset_index()
    )
    display(kpis.head())

**Exercise:** Calculate total revenue per `customer_id`.

## 8️⃣ Grouping & Aggregation
**Concepts:** `groupby()`, `.sum()`, `.mean()`, `.agg()`

**Business Context:** Create monthly/category rollups like a pivot table.

In [ ]:
if 'orders' in globals() and 'order_items' in globals():
    tmp = pd.merge(orders_clean[['order_id','order_date','order_status']], order_items, on='order_id', how='inner')
    tmp['revenue'] = tmp['quantity'] * tmp['item_price']
    tmp['month'] = pd.to_datetime(tmp['order_date']).dt.to_period('M')
    monthly_revenue = tmp.groupby('month')['revenue'].sum().reset_index().sort_values('month')
    display(monthly_revenue.head(12))

**Exercise:** Group total quantity sold by `product_id` (use the merged frame with items).

## 9️⃣ Sorting & Ranking
**Concepts:** `.sort_values()`, `.nlargest()`

**Business Context:** Find top products/customers by revenue.

In [ ]:
if 'orders' in globals() and 'order_items' in globals():
    tmp = pd.merge(orders_clean[['order_id','order_date']], order_items, on='order_id', how='inner')
    tmp['revenue'] = tmp['quantity'] * tmp['item_price']
    top_products = (
        tmp.groupby('product_id')['revenue']
        .sum()
        .reset_index()
        .sort_values('revenue', ascending=False)
        .head(5)
    )
    display(top_products)

**Exercise:** Find top 3 customers by total spend (hint: group by `customer_id`).

## 🔟 Exporting Results
**Concepts:** `.to_csv()`

**Business Context:** Save clean/aggregated datasets for Power BI or dashboards.

In [ ]:
from pathlib import Path
if 'monthly_revenue' in globals():
    outdir = Path('../data/processed')
    outdir.mkdir(parents=True, exist_ok=True)
    monthly_revenue.to_csv(outdir / 'monthly_revenue.csv', index=False)
    print('Saved: data/processed/monthly_revenue.csv')

## ⭐ Mini Project – Monthly KPI Report
**Goal:**
1) Load and clean data
2) Merge orders + items + customers
3) Compute order revenue, items
4) Aggregate to monthly level (orders, revenue, AOV)
5) Export results for visualization

**Starter Code:**

In [ ]:
# 1) Load and clean
orders = pd.read_csv(ORDERS_PATH)
order_items = pd.read_csv(ORDER_ITEMS_PATH)
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')
orders = orders.drop_duplicates()

# 2) Merge & compute revenue
merged = pd.merge(orders[['order_id','customer_id','order_date','order_status']],
                  order_items[['order_id','product_id','quantity','item_price']],
                  on='order_id', how='inner')
merged['revenue'] = merged['quantity'] * merged['item_price']

# 3) Filter recognized revenue statuses
recognized = merged[merged['order_status'].isin(['paid','shipped','delivered'])].copy()

# 4) Monthly KPIs
recognized['month'] = recognized['order_date'].dt.to_period('M')
monthly = recognized.groupby('month').agg(
    orders=('order_id','nunique'),
    revenue=('revenue','sum')
).reset_index()
monthly['aov'] = monthly['revenue'] / monthly['orders']
monthly